# 74 Job · Orquestación y Monitoreo

Vas a crear, ejecutar y monitorear un DAG completo con el Databricks SDK. Después de crearlo, **abrí el job y recorré visualmente cada dependencia, parámetro y condición**. El laboratorio cubre tareas comunes, dependencias, retries, condiciones, loops, `run_if`, monitoreo y troubleshooting.

In [0]:
CATALOG = "big_data_ii_2025"
SCHEMA  = "spark_examples"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Trabajando en {CATALOG}.{SCHEMA}")

In [0]:
from databricks.sdk import WorkspaceClient

# Crea un cliente autenticado con la sesión actual y resuelve rutas sin usuarios hardcodeados.
w = WorkspaceClient()
usuario = w.current_user.me().user_name
BASE = f"/Workspace/Users/{usuario}/spark_databricks"
NOMBRE_JOB = f"Semana 13 · ETL MovieLens · {usuario}"
print(f"Usuario: {usuario}")
print(f"Ruta esperada de notebooks: {BASE}")

## DAG que vamos a construir

```text
                     ┌──────────────────────┐
                     │  01_ingesta_bronze   │  notebook · max_retries=2
                     └──────────┬───────────┘
                                │  task value: filas_ingeridas
                     ┌──────────▼───────────┐
                     │ 02_control_calidad   │  notebook
                     └──────────┬───────────┘
                                │  task value: filas_malas
                     ┌──────────▼───────────┐
                     │  03_gate_calidad     │  if/else · filas_malas == 0
                     └────┬─────────────┬───┘
                  true    │             │ false
             ┌────────────▼───┐   ┌─────▼──────────────┐
             │ 04_for_each    │   │ 05_registrar_falla │
             │ género · c=2   │   │ notebook 73        │
             └────────┬───────┘   └─────┬──────────────┘
                      │                 │
                 ┌────▼─────────────────▼────┐
                 │      06_gold_y_cierre     │  run_if: ALL_DONE
                 └───────────────────────────┘
```

In [0]:
from databricks.sdk.service.jobs import (
    ConditionTask, ConditionTaskOp, ForEachTask, JobParameterDefinition,
    NotebookTask, RunIf, Task, TaskDependency
)

# Reemplazar el job homónimo hace esta celda idempotente para cada estudiante.
existentes = list(w.jobs.list(name=NOMBRE_JOB))
for existente in existentes:
    print(f"Eliminando job anterior {existente.job_id}...")
    w.jobs.delete(job_id=existente.job_id)

tareas = [
    Task(
        task_key="01_ingesta_bronze",
        notebook_task=NotebookTask(
            notebook_path=f"{BASE}/70_Job_01_Ingesta_Bronze",
            base_parameters={
                "entorno": "{{job.parameters.entorno}}",
                "limite_filas": "{{job.parameters.limite_filas}}"
            }),
        max_retries=2,
        min_retry_interval_millis=30_000),
    Task(
        task_key="02_control_calidad",
        depends_on=[TaskDependency(task_key="01_ingesta_bronze")],
        notebook_task=NotebookTask(
            notebook_path=f"{BASE}/71_Job_02_Control_Calidad",
            base_parameters={"inyectar_error": "{{job.parameters.inyectar_error}}"})),
    Task(
        task_key="03_gate_calidad",
        depends_on=[TaskDependency(task_key="02_control_calidad")],
        condition_task=ConditionTask(
            op=ConditionTaskOp.EQUAL_TO,
            left="{{tasks.02_control_calidad.values.filas_malas}}",
            right="0")),
    Task(
        task_key="04_metricas_por_genero",
        depends_on=[TaskDependency(task_key="03_gate_calidad", outcome="true")],
        for_each_task=ForEachTask(
            inputs='["Drama","Comedy","Action","Thriller"]',
            concurrency=2,
            task=Task(
                task_key="metrica_genero",
                notebook_task=NotebookTask(
                    notebook_path=f"{BASE}/72_Job_03_ForEach_Genero",
                    base_parameters={"genero": "{{input}}"})))),
    Task(
        task_key="05_registrar_falla",
        depends_on=[TaskDependency(task_key="03_gate_calidad", outcome="false")],
        notebook_task=NotebookTask(
            notebook_path=f"{BASE}/73_Job_04_Gold_y_Cierre",
            base_parameters={
                "accion": "registrar_falla",
                "run_id": "{{job.run_id}}"
            })),
    Task(
        task_key="06_gold_y_cierre",
        depends_on=[
            TaskDependency(task_key="04_metricas_por_genero"),
            TaskDependency(task_key="05_registrar_falla")
        ],
        run_if=RunIf.ALL_DONE,
        notebook_task=NotebookTask(
            notebook_path=f"{BASE}/73_Job_04_Gold_y_Cierre",
            base_parameters={
                "accion": "cierre",
                "run_id": "{{job.run_id}}"
            }))
]

job = w.jobs.create(
    name=NOMBRE_JOB,
    tasks=tareas,
    parameters=[
        JobParameterDefinition(name="entorno", default="dev"),
        JobParameterDefinition(name="limite_filas", default="50000"),
        JobParameterDefinition(name="inyectar_error", default="false")
    ])
job_id = job.job_id
print(f"Job creado: {job_id}")

In [0]:
# Construye un enlace directo para inspeccionar el DAG recién creado en la UI.
enlace_job = f"{w.config.host}/jobs/{job_id}"
print(f"Abrí el job en: {enlace_job}")

In [0]:
from datetime import timedelta

# Dispara una corrida de referencia usando la rama de calidad exitosa.
print("Iniciando una corrida sin errores inyectados...")
operacion_corrida = w.jobs.run_now(
    job_id=job_id,
    job_parameters={
        "entorno": "dev",
        "limite_filas": "50000",
        "inyectar_error": "false"
    })
# Espera el estado terminal con un límite para evitar una espera indefinida.
resultado_corrida = operacion_corrida.result(timeout=timedelta(minutes=20))
run_id = resultado_corrida.run_id
print(f"Corrida {run_id} terminada. Revisá el detalle en {w.config.host}/jobs/{job_id}/runs/{run_id}")

In [0]:
from pyspark.sql import Row

# Recorre las últimas corridas con sus tareas expandidas para reproducir el run history.
filas_monitoreo = []
for corrida in w.jobs.list_runs(job_id=job_id, limit=5, expand_tasks=True):
    for tarea in corrida.tasks or []:
        # Normaliza tiempos y estados que pueden venir vacíos en tareas omitidas.
        inicio = tarea.start_time or 0
        fin = tarea.end_time or inicio
        estado = (tarea.state.result_state.value if tarea.state and tarea.state.result_state
                  else tarea.state.life_cycle_state.value if tarea.state and tarea.state.life_cycle_state
                  else "UNKNOWN")
        filas_monitoreo.append(Row(
            run_id=int(corrida.run_id),
            task_key=tarea.task_key,
            state=estado,
            duracion_segundos=round((fin - inicio) / 1000, 2),
            attempt_number=int(tarea.attempt_number or 0)))

# Convierte la respuesta del SDK en un DataFrame ordenado y fácil de comparar.
if filas_monitoreo:
    display(spark.createDataFrame(filas_monitoreo).orderBy("run_id", "task_key"))
else:
    print("Todavía no hay tareas en el historial de corridas.")

## Ejercicio guiado en la UI

1. Abrí **Jobs & Pipelines**, entrá al job recién creado y recorré su DAG.
2. Cambiá el job parameter `inyectar_error` a `true`, ejecutá otra corrida y observá cómo se activa la rama `false`.
3. Provocá o seleccioná una tarea fallida, usá **Repair run** y fijate que solo se vuelven a ejecutar las tareas afectadas.
4. Agregá un trigger **Scheduled** con cron `0 0 6 * * ?`, zona `UTC`, y dejalo **pausado**.
5. Abrí **Runs**, cambiá a la vista matriz y compará duraciones e intentos entre corridas.

## Equivalente declarativo con Databricks Asset Bundles

Este YAML es lectura de referencia; no se despliega desde Free Edition. Los targets permiten conservar la misma definición y cambiar parámetros por ambiente.

```yaml
bundle:
  name: bcd8216-clase13

variables:
  entorno:
    default: dev
  limite_filas:
    default: "50000"
  inyectar_error:
    default: "false"

resources:
  jobs:
    etl_movielens:
      name: BCD8216-Clase13-ETL-MovieLens-${bundle.target}
      parameters:
        - name: entorno
          default: ${var.entorno}
        - name: limite_filas
          default: ${var.limite_filas}
        - name: inyectar_error
          default: ${var.inyectar_error}
      tasks:
        - task_key: 01_ingesta_bronze
          max_retries: 2
          min_retry_interval_millis: 30000
          notebook_task:
            notebook_path: ./70_Job_01_Ingesta_Bronze.ipynb
            base_parameters:
              entorno: "{{job.parameters.entorno}}"
              limite_filas: "{{job.parameters.limite_filas}}"
        - task_key: 02_control_calidad
          depends_on:
            - task_key: 01_ingesta_bronze
          notebook_task:
            notebook_path: ./71_Job_02_Control_Calidad.ipynb
            base_parameters:
              inyectar_error: "{{job.parameters.inyectar_error}}"
        - task_key: 03_gate_calidad
          depends_on:
            - task_key: 02_control_calidad
          condition_task:
            op: EQUAL_TO
            left: "{{tasks.02_control_calidad.values.filas_malas}}"
            right: "0"
        - task_key: 04_metricas_por_genero
          depends_on:
            - task_key: 03_gate_calidad
              outcome: "true"
          for_each_task:
            inputs: '["Drama","Comedy","Action","Thriller"]'
            concurrency: 2
            task:
              task_key: metrica_genero
              notebook_task:
                notebook_path: ./72_Job_03_ForEach_Genero.ipynb
                base_parameters:
                  genero: "{{input}}"
        - task_key: 05_registrar_falla
          depends_on:
            - task_key: 03_gate_calidad
              outcome: "false"
          notebook_task:
            notebook_path: ./73_Job_04_Gold_y_Cierre.ipynb
            base_parameters:
              accion: registrar_falla
              run_id: "{{job.run_id}}"
        - task_key: 06_gold_y_cierre
          depends_on:
            - task_key: 04_metricas_por_genero
            - task_key: 05_registrar_falla
          run_if: ALL_DONE
          notebook_task:
            notebook_path: ./73_Job_04_Gold_y_Cierre.ipynb
            base_parameters:
              accion: cierre
              run_id: "{{job.run_id}}"

targets:
  dev:
    default: true
    variables:
      entorno: dev
      limite_filas: "50000"
  prod:
    variables:
      entorno: prod
      limite_filas: "100000"
```

In [0]:
# Limpieza opcional: quitá el comentario solo si querés eliminar el job del workspace.
# w.jobs.delete(job_id=job_id)
# print(f"Job {job_id} eliminado.")

## Cierre

- Creaste idempotentemente un Lakeflow Job serverless con seis tareas.
- Aplicaste retries, `if/else`, `For each` con concurrencia limitada y `ALL_DONE`.
- Ejecutaste el DAG con parámetros a nivel de job.
- Construiste una vista programática equivalente al *run history*.
- Relacionaste la definición SDK con la UI y con Databricks Asset Bundles.